# Deep Learning Lab: Image Classification & Feature Visualization

This notebook covers:
- Loading custom datasets (faces & flowers)
- Using pretrained models (ResNet50)
- Top-1 and Top-5 predictions
- Feature extraction
- PCA, t-SNE, UMAP visualization


In [ ]:
!pip install umap-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.imagenet_utils import preprocess_input, decode_predictions
from tensorflow.keras.models import Model

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

In [ ]:
dataset_path = '/content/drive/MyDrive/dataset'
IMG_SIZE = (224, 224)

def load_images(folder):
    images = []
    labels = []
    
    for label in os.listdir(folder):
        path = os.path.join(folder, label)
        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            img = image.load_img(img_path, target_size=IMG_SIZE)
            img = image.img_to_array(img)
            images.append(img)
            labels.append(label)
    
    return np.array(images), np.array(labels)

face_images, face_labels = load_images(dataset_path + '/faces')
flower_images, flower_labels = load_images(dataset_path + '/flowers')

print(face_images.shape, flower_images.shape)

In [ ]:
model = ResNet50(weights='imagenet')

In [ ]:
def predict_images(images):
    imgs = preprocess_input(images.copy())
    preds = model.predict(imgs)
    decoded = decode_predictions(preds, top=5)
    
    for i, pred in enumerate(decoded[:5]):
        print(f"\nImage {i+1}")
        print("Top-1:", pred[0])
        print("Top-5:")
        for p in pred:
            print(p)

predict_images(face_images)
predict_images(flower_images)

In [ ]:
feature_model = Model(inputs=model.input, outputs=model.layers[-2].output)

def extract_features(images):
    imgs = preprocess_input(images.copy())
    return feature_model.predict(imgs)

face_features = extract_features(face_images)
flower_features = extract_features(flower_images)

In [ ]:
def reduce_features(features, method='pca'):
    if method == 'pca':
        return PCA(n_components=2).fit_transform(features)
    elif method == 'tsne':
        return TSNE(n_components=2).fit_transform(features)
    elif method == 'umap':
        return umap.UMAP(n_components=2).fit_transform(features)

In [ ]:
def plot_2d(data, labels, title):
    plt.figure()
    for label in np.unique(labels):
        idx = labels == label
        plt.scatter(data[idx,0], data[idx,1], label=label)
    plt.legend()
    plt.title(title)
    plt.show()

face_pca = reduce_features(face_features, 'pca')
plot_2d(face_pca, face_labels, 'Face PCA')

face_tsne = reduce_features(face_features, 'tsne')
plot_2d(face_tsne, face_labels, 'Face t-SNE')

face_umap = reduce_features(face_features, 'umap')
plot_2d(face_umap, face_labels, 'Face UMAP')